In [0]:
import mlflow
from openai import OpenAI

In [0]:
mlflow.openai.autolog()

In [0]:
# set an experiment
experiment = mlflow.set_experiment(experiment_name="/Workspace/Users/biswadeep.upadhyay@databricks.com/mlflow_artifacts/mlflow-tracing-101")
mlflow.set_experiment_tags({"owner": "beepz"})

In [0]:
experiment.experiment_id

In [0]:
base_url = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}/serving-endpoints'
databricks_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [0]:
client = OpenAI(
    api_key=databricks_token,
    base_url=base_url
  )

In [0]:
# basic tracing
completion = client.chat.completions.create(
  model = 'databricks-gpt-5-1',
  messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {
      "role": "user",
      "content": "What is MLflow tracing?"
    }
  ],
  temperature=0.1
)

In [0]:
print(completion.choices[0].message.content)

In [0]:
# formatted output

from pydantic import BaseModel

class EventDetails(BaseModel):
    speakers: str
    date: str
    num_attendees: int
    description: str

raw_event_details = """137 views  Streamed live on Mar 6, 2025
Stop fighting with model signatures and focus on your model logic! MLflow 2.20.0 lets you use Python's native type annotations to automatically validate inputs and infer model signatures. MLflow Software Engineer, Serena Ruan, will show you how this addresses some of the most common pain points in custom model development and simplifies your workflow with patterns for everything from simple data types to complex Pydantic models.

Come see how these improvements make your MLflow experience more Pythonic and robust, with fewer runtime surprises. Bring your questions for our Q&A session!"""


completion = client.chat.completions.parse(
  model = 'databricks-gpt-5-1',
  messages = [
    {
      "role": "system", "content": "Extract the even informations."
    },
    {
      "role": "user",
      "content": raw_event_details
    }
  ],
  response_format=EventDetails
)

In [0]:
print(completion.choices[0].message.content)

In [0]:
# organizing traces
with mlflow.start_run() as run:
    raw_event_details_1 = """137 views  Streamed live on Mar 6, 2025
Stop fighting with model signatures and focus on your model logic! MLflow 2.20.0 lets you use Python's native type annotations to automatically validate inputs and infer model signatures. MLflow Software Engineer, Serena Ruan, will show you how this addresses some of the most common pain points in custom model development and simplifies your workflow with patterns for everything from simple data types to complex Pydantic models.

Come see how these improvements make your MLflow experience more Pythonic and robust, with fewer runtime surprises. Bring your questions for our Q&A session!"""

    raw_event_details_2 = """40 views  
Streamed live Mar 26, 2025
This community meetup will focus on 2 major updates:

🚀 MLflow Prompt Registry - As large language models (LLMs) become integral to AI workflows, MLflow is expanding its capabilities to support prompt engineering. Learn how the new MLflow Prompt Registry enables seamless tracking, versioning, and experimentation with prompts—making it easier to manage LLM-based applications.

📦 MLflow 3: A Model-Centric Approach - MLflow 3 introduces a more structured, model-first approach to managing the ML lifecycle. This new design enhances how models are tracked, stored, and deployed, making workflows more intuitive and scalable. Learn what’s changing and how these improvements will help teams streamline their machine learning operations.

Harutaka Kawamura and Yuki Watanabe will walk through these latest developments and answer your MLflow questions in a live Q&A session!"""

    for e in [raw_event_details_1, raw_event_details_2]:
        completion = client.chat.completions.parse(
            model="databricks-gpt-5-1",
            messages=[
                {"role": "system", "content": "Extract the event information."},
                {"role": "user", "content": e},
            ],
            response_format=EventDetails,
        )

        print(completion)